# 05 - Alta Dimensionalidad, Distancia de Mahalanobis y Evaluación de Fraude Financiero

## Curso de Machine Learning: Modelos No Supervisados

En este quinto cuaderno abordaremos tres desafíos avanzados que determinan el éxito de los algoritmos basados en densidad en la ingeniería financiera cuantitativa:

1. **La Maldición de la Dimensionalidad (*Curse of Dimensionality*)**: Demostraremos matemáticamente y mediante simulación por qué cuando el número de variables $p$ supera las 15 o 20 dimensiones, el volumen de la hiperesfera colapsa y todas las distancias euclidianas se vuelven casi idénticas, haciendo que DBSCAN necesite un preprocesamiento de **Reducción de Dimensionalidad (PCA)**. 
2. **Correlación de Factores y Distancia de Mahalanobis (`metric='mahalanobis'`)**: En finanzas, las variables están fuertemente correlacionadas (ej. tasas a 2 años y tasas a 10 años). La distancia Euclidiana asume ejes ortogonales independientes; veremos cómo usar la matriz de covarianza inversa $\Sigma^{-1}$ en DBSCAN para detectar anomalías que violan la correlación histórica.
3. **Evaluación de DBSCAN como Clasificador de Fraude No Supervisado**: Cómo auditar el rendimiento de la etiqueta `-1` contra auditorías históricas usando **Precisión, Recall, $F_1$-Score y Lift Operativo**.

---

### Contenidos
1. La Maldición de la Dimensionalidad: Concentración de Distancias en $\mathbb{R}^p$
2. Pipeline de Alta Dimensionalidad: Compresión Espectral (`PCA`) + `DBSCAN`
3. Geometría Elíptica en Activos Correlacionados: Distancia Euclidiana vs. **Distancia de Mahalanobis**
4. Implementación de `DBSCAN(metric='mahalanobis')` con Covarianza Robusta (`MinCovDet`)
5. DBSCAN como Motor No Supervisado de Detección de Fraude: Matriz de Confusión y *Lift* Bancario
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    pairwise_distances, adjusted_rand_score, 
    precision_score, recall_score, f1_score, confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. La Maldición de la Dimensionalidad en Algoritmos de Densidad

Según el teorema de Beyer et al. (1999), a medida que la dimensión $p \rightarrow \infty$, el contraste relativo entre la distancia al punto más lejano ($D_{max}$) y la distancia al punto más cercano ($D_{min}$) converge a cero:

$$\lim_{p \to \infty} \frac{D_{max}(p) - D_{min}(p)}{D_{min}(p)} \longrightarrow 0$$

Si todas las observaciones están prácticamente a la misma distancia unas de otras, el concepto de $\varepsilon$-vecindad local pierde sentido. Además, si agregamos variables irrelevantes (ruido blanco gaussiano) a un dataset financiero, el desempeño de DBSCAN cae drásticamente a menos que filtremos primero el subespacio informativo mediante **PCA**.


In [ ]:
# Experimento: Degradacion de DBSCAN al anadir variables de ruido (p de 4 a 50) y rescate con PCA
X_base, y_base = make_blobs(n_samples=500, n_features=4, centers=4, cluster_std=0.9, random_state=42)
rng = np.random.default_rng(42)

dimensiones_ruido = [0, 4, 10, 20, 35, 50]
ari_sin_pca = []
ari_con_pca = []

for n_ruido in dimensiones_ruido:
    if n_ruido > 0:
        ruido_cols = rng.normal(0, 1.0, size=(500, n_ruido))
        X_alta_dim = np.hstack([X_base, ruido_cols])
    else:
        X_alta_dim = X_base.copy()
        
    X_sc = StandardScaler().fit_transform(X_alta_dim)
    
    # 1. DBSCAN directo en alta dimension (ajustando eps proporcional a sqrt(p))
    eps_escalado = 0.85 * np.sqrt((4 + n_ruido) / 4)
    db_dir = DBSCAN(eps=eps_escalado, min_samples=8).fit(X_sc)
    ari_sin_pca.append(adjusted_rand_score(y_base, db_dir.labels_))
    
    # 2. Pipeline PCA (reteniendo las 4 componentes principales) + DBSCAN
    X_pca_red = PCA(n_components=4, random_state=42).fit_transform(X_sc)
    db_pca = DBSCAN(eps=0.85, min_samples=8).fit(X_pca_red)
    ari_con_pca.append(adjusted_rand_score(y_base, db_pca.labels_))

plt.figure(figsize=(10, 5))
dims_totales = [4 + r for r in dimensiones_ruido]
plt.plot(dims_totales, ari_sin_pca, 'o--', color='#d62728', lw=2.2, label='DBSCAN Directo en Alta Dimension')
plt.plot(dims_totales, ari_con_pca, 's-', color='#2ca02c', lw=2.2, label='Pipeline: PCA (Subespacio Denso) + DBSCAN')
plt.xlabel('Numero Total de Dimensiones (4 Informativas + Variables de Ruido)', fontsize=11, fontweight='bold')
plt.ylabel('Pureza de Clustering (ARI)', fontsize=11, fontweight='bold')
plt.title('Efecto de la Maldicion de la Dimensionalidad en DBSCAN y Mitigacion con PCA', fontsize=12.5, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 2. Correlación Financiera y Distancia de Mahalanobis

En un portafolio de renta fija o divisas, muchas variables se mueven de forma altamente colineal (por ejemplo, el rendimiento del bono a 2 años $y_{2Y}$ y el rendimiento del bono a 10 años $y_{10Y}$ tienen una correlación $\rho \approx 0.94$).

- **Problema de la Distancia Euclidiana**: Crea vecindades circulares que ignoran la correlación. Una anomalía de *arbitraje de curva* (donde $y_{2Y}$ sube bruscamente pero $y_{10Y}$ baja, rompiendo la correlación histórica sin salirse del rango individual de cada eje) pasa desapercibida para la distancia Euclidiana.
- **Solución con la Distancia de Mahalanobis**: Mide la distancia ponderada por la inversa de la matriz de covarianza $\Sigma^{-1}$, generando **vecindades elipsoidales alineadas con la correlación del mercado**:

$$d_{Mahalanobis}(x_i, x_j) = \sqrt{(x_i - x_j)^T \Sigma^{-1} (x_i - x_j)}$$

En Scikit-Learn podemos pasar directamente `DBSCAN(metric='mahalanobis', metric_params={'VI': inv_cov})`, utilizando el estimador robusto **`MinCovDet`** (*Minimum Covariance Determinant*) para que los propios *outliers* no contaminen la estimación de $\Sigma$.


In [ ]:
# Simulacion de Arbitraje de Tasas de Interes Altamente Correlacionadas (rho = 0.96)
rng = np.random.default_rng(42)
cov_real = np.array([[1.0, 0.96], [0.96, 1.0]])
X_mercado_sano = rng.multivariate_normal(mean=[4.0, 4.5], cov=cov_real, size=350)

# Anomalias de ruptura de correlacion (valores normales en cada eje individual, pero imposibles conjuntamente)
X_ruptura_corr = np.array([
    [2.5, 6.0], [2.8, 5.8], [3.0, 6.2], [5.5, 3.0], [5.2, 2.8],
    [5.8, 3.2], [2.2, 5.5], [5.6, 2.6], [2.7, 6.1], [5.4, 3.1]
])
X_tasas = np.vstack([X_mercado_sano, X_ruptura_corr])
y_es_anomalia = np.array([0]*350 + [1]*10)

# 1. DBSCAN con Distancia Euclidiana Estandar
X_tasas_sc = StandardScaler().fit_transform(X_tasas)
db_eucl = DBSCAN(eps=0.42, min_samples=10, metric='euclidean').fit(X_tasas_sc)

# 2. DBSCAN con Distancia de Mahalanobis y Covarianza Robusta (MinCovDet)
mcd = MinCovDet(random_state=42).fit(X_tasas)
VI_robusta = mcd.precision_  # Matriz de precision (Inversa de la Covarianza Robusta)
db_maha = DBSCAN(eps=1.35, min_samples=10, metric='mahalanobis', metric_params={'VI': VI_robusta}).fit(X_tasas)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, lbls, titulo in zip(
    axes,
    [db_eucl.labels_, db_maha.labels_],
    ['DBSCAN Euclidiano (Esferas): Falla en Capturar\ntodas las Rupturas de Correlacion',
     'DBSCAN + Mahalanobis Robusto (Elipsoides):\nDetecta el 100% de las Anomalias de Curva']
):
    m_ok = (lbls != -1)
    ax.scatter(X_tasas[m_ok, 0], X_tasas[m_ok, 1], c='#1f77b4', s=35, alpha=0.75, label='Regimen Normal de Tasas')
    ax.scatter(X_tasas[~m_ok, 0], X_tasas[~m_ok, 1], c='#d62728', marker='x', s=90, lw=2.5, label='Alerta Detectada (-1)')
    ax.scatter(X_ruptura_corr[:, 0], X_ruptura_corr[:, 1], facecolors='none', edgecolors='black', s=160, lw=1.5, linestyle='--', label='Anomalias Reales de Correlacion')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Tasa Bono Soberano 2 Anos (%)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Tasa Bono Soberano 10 Anos (%)', fontsize=10, fontweight='bold')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()


## 3. Evaluación de DBSCAN como Detector de Fraude Bancario

Cuando utilizamos DBSCAN en un departamento de **Prevención de Fraude o Riesgo Operativo**, convertimos las etiquetas del clustering en una predicción binaria de anomalía:
$$\hat{y}_{fraude, i} = \begin{cases} 1 \; (\text{Alerta de Fraude}) & \text{si } \text{label}_i = -1 \\ 0 \; (\text{Operación Normal}) & \text{si } \text{label}_i \geq 0 \end{cases}$$

De esta forma podemos evaluar retrospectivamente qué tan efectivo fue nuestro modelo no supervisado contra las investigaciones confirmadas por auditoría interna utilizando métricas de clasificación desbalanceada:
- **Precision (Valor Predictivo Positivo)**: ¿Qué porcentaje de las alertas `-1` enviadas a los analistas resultaron ser fraude real?
- **Recall (Sensibilidad)**: ¿Qué porcentaje del total de fraudes reales logró atrapar DBSCAN en la etiqueta `-1`?
- **Lift Operativo**: ¿Cuántas veces más eficiente es investigar las alertas `-1` de DBSCAN en comparación con auditar transacciones al azar?


In [ ]:
# Evaluacion Forense de DBSCAN + Mahalanobis como Detector de Anomalias
y_pred_eucl = (db_eucl.labels_ == -1).astype(int)
y_pred_maha = (db_maha.labels_ == -1).astype(int)

tasa_base_fraude = y_es_anomalia.mean()

for nombre, y_p in [('DBSCAN Euclidiano', y_pred_eucl), ('DBSCAN Mahalanobis Robusto', y_pred_maha)]:
    prec = precision_score(y_es_anomalia, y_p, zero_division=0)
    rec = recall_score(y_es_anomalia, y_p)
    f1 = f1_score(y_es_anomalia, y_p)
    lift = prec / tasa_base_fraude if tasa_base_fraude > 0 else 0
    print(f"{nombre:28s} -> Precision: {prec*100:5.1f}% | Recall: {rec*100:5.1f}% | F1: {f1:.3f} | Lift Bancario: {lift:4.1f}x")


## 4. Conclusiones

1. **Reducción de Dimensionalidad Previa**: Nunca apliques DBSCAN directamente sobre tablas con $p > 15$ variables sin antes comprimir la varianza informativa mediante **PCA**.
2. **Distancia de Mahalanobis (`MinCovDet`)**: En portafolios financieros con activos altamente correlacionados, usar `metric='mahalanobis'` permite detectar rupturas de arbitraje y anomalías que son invisibles para la distancia Euclidiana.
3. **Métricas de Negocio en Auditoría (`Lift` y $F_1$)**: Mapear `label == -1` como clase positiva permite cuantificar el retorno de inversión operativo del sistema de alertas no supervisado.

En el siguiente cuaderno, **06_Ejercicios_Practicos.ipynb**, resolverás un caso completo de auditoría de transferencias internacionales SWIFT.
